# 00 — Environment and File Check

**Purpose:** Confirm that this repository can find the Little Plover River model files, PyCAP input files, important Python packages, and command-line executables before we try to run MODFLOW 6 or PyCAP.

This notebook does **not** run the groundwater model yet. It only checks that the project structure is ready.

Expected project folders:

```text
models/lpr_mf6/steady_state/
data/raw/lpr_pycap/pycap_base/
data/raw/lpr_pycap/inputs/
scripts/lpr_legacy/
```


## 1. Find the project root

This cell is written so the notebook works whether Jupyter starts in the repository root or inside the `notebooks/` folder.


In [1]:
from pathlib import Path
import os
import sys
import json
import shutil
import importlib
import subprocess
from datetime import datetime

def find_project_root(start=None):
    """Find the repository root by looking for common project marker files."""
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start] + list(start.parents)

    for candidate in candidates:
        if (candidate / ".git").exists() and (candidate / "environment.yml").exists():
            return candidate

    for candidate in candidates:
        if (candidate / "models").exists() and (candidate / "data").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. Try opening Jupyter from the Modeling-Uncertainties repo root."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

PROJECT_ROOT


PosixPath('/workspaces/Modeling-Uncertainties')

## 2. Define important project paths

These are the main folders we will use throughout the project.


In [2]:
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

LPR_MF6_DIR = MODELS_DIR / "lpr_mf6" / "steady_state"
LPR_PYCAP_RAW_DIR = DATA_DIR / "raw" / "lpr_pycap"
LPR_PYCAP_BASE_DIR = LPR_PYCAP_RAW_DIR / "pycap_base"
LPR_PYCAP_INPUTS_DIR = LPR_PYCAP_RAW_DIR / "inputs"
LPR_LEGACY_SCRIPTS_DIR = SCRIPTS_DIR / "lpr_legacy"

for p in [
    PROJECT_ROOT,
    DATA_DIR,
    MODELS_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    NOTEBOOKS_DIR,
    SCRIPTS_DIR,
    LPR_MF6_DIR,
    LPR_PYCAP_BASE_DIR,
    LPR_PYCAP_INPUTS_DIR,
    LPR_LEGACY_SCRIPTS_DIR,
]:
    print(f"{p.name:30s} -> {p}")


Modeling-Uncertainties         -> /workspaces/Modeling-Uncertainties
data                           -> /workspaces/Modeling-Uncertainties/data
models                         -> /workspaces/Modeling-Uncertainties/models
results                        -> /workspaces/Modeling-Uncertainties/results
figures                        -> /workspaces/Modeling-Uncertainties/figures
notebooks                      -> /workspaces/Modeling-Uncertainties/notebooks
scripts                        -> /workspaces/Modeling-Uncertainties/scripts
steady_state                   -> /workspaces/Modeling-Uncertainties/models/lpr_mf6/steady_state
pycap_base                     -> /workspaces/Modeling-Uncertainties/data/raw/lpr_pycap/pycap_base
inputs                         -> /workspaces/Modeling-Uncertainties/data/raw/lpr_pycap/inputs
lpr_legacy                     -> /workspaces/Modeling-Uncertainties/scripts/lpr_legacy


## 3. Check that required folders exist

This gives a quick pass/fail check for the project layout.


In [3]:
import pandas as pd

required_dirs = {
    "Project root": PROJECT_ROOT,
    "MODFLOW 6 steady-state model": LPR_MF6_DIR,
    "PyCAP baseline folder": LPR_PYCAP_BASE_DIR,
    "PyCAP inputs folder": LPR_PYCAP_INPUTS_DIR,
    "Legacy scripts folder": LPR_LEGACY_SCRIPTS_DIR,
    "Results folder": RESULTS_DIR,
    "Figures folder": FIGURES_DIR,
}

dir_check = pd.DataFrame(
    [
        {
            "item": name,
            "path": str(path.relative_to(PROJECT_ROOT)) if path.is_relative_to(PROJECT_ROOT) else str(path),
            "exists": path.exists(),
            "type": "directory" if path.is_dir() else "missing",
        }
        for name, path in required_dirs.items()
    ]
)

dir_check


,item,path,exists,type
0,Project root,.,True,directory
1,MODFLOW 6 steady-state model,models/lpr_mf6/steady_state,True,directory
2,PyCAP baseline folder,data/raw/lpr_pycap/pycap_base,True,directory
3,PyCAP inputs folder,data/raw/lpr_pycap/inputs,True,directory
4,Legacy scripts folder,scripts/lpr_legacy,True,directory
5,Results folder,results,True,directory
6,Figures folder,figures,True,directory


## 4. Check important MODFLOW 6 files

These files should exist inside:

```text
models/lpr_mf6/steady_state/
```

We are checking for the main simulation name file, groundwater-flow model name file, and major package files.


In [4]:
required_mf6_files = [
    "mfsim.nam",
    "lpr_ss.tdis",
    "lpr_ss.ims",
    "lpr_ss_gwf.nam",
    "lpr_ss_gwf.dis",
    "lpr_ss_gwf.npf",
    "lpr_ss_gwf.ic",
    "lpr_ss_gwf.oc",
    "lpr_ss_gwf.rcha",
    "lpr_ss_gwf.wel",
    "lpr_ss_gwf.drn",
    "lpr_ss_gwf.chd",
    "lpr_ss.sfr",
    "sfr_obs.obs",
]

mf6_file_check = pd.DataFrame(
    [
        {
            "file": filename,
            "relative_path": str((LPR_MF6_DIR / filename).relative_to(PROJECT_ROOT)),
            "exists": (LPR_MF6_DIR / filename).exists(),
            "size_MB": round((LPR_MF6_DIR / filename).stat().st_size / 1_000_000, 3)
            if (LPR_MF6_DIR / filename).exists()
            else None,
        }
        for filename in required_mf6_files
    ]
)

mf6_file_check


,file,relative_path,exists,size_MB
0,mfsim.nam,models/lpr_mf6/steady_state/mfsim.nam,True,0.000
1,lpr_ss.tdis,models/lpr_mf6/steady_state/lpr_ss.tdis,True,0.000
2,lpr_ss.ims,models/lpr_mf6/steady_state/lpr_ss.ims,True,0.001
3,lpr_ss_gwf.nam,models/lpr_mf6/steady_state/lpr_ss_gwf.nam,True,0.000
4,lpr_ss_gwf.dis,models/lpr_mf6/steady_state/lpr_ss_gwf.dis,True,0.001
5,lpr_ss_gwf.npf,models/lpr_mf6/steady_state/lpr_ss_gwf.npf,True,0.001
6,lpr_ss_gwf.ic,models/lpr_mf6/steady_state/lpr_ss_gwf.ic,True,0.000
7,lpr_ss_gwf.oc,models/lpr_mf6/steady_state/lpr_ss_gwf.oc,True,0.000
8,lpr_ss_gwf.rcha,models/lpr_mf6/steady_state/lpr_ss_gwf.rcha,True,0.000
9,lpr_ss_gwf.wel,models/lpr_mf6/steady_state/lpr_ss_gwf.wel,True,0.000


## 5. Check important PyCAP files

These files should exist inside:

```text
data/raw/lpr_pycap/
```


In [5]:
required_pycap_files = [
    LPR_PYCAP_BASE_DIR / "LPR_Redux.yml",
    LPR_PYCAP_BASE_DIR / "depletion_potential.json",
    LPR_PYCAP_BASE_DIR / "output" / "LPR_Redux.report.txt",
    LPR_PYCAP_BASE_DIR / "output" / "LPR_Redux.table_report.csv",
    LPR_PYCAP_BASE_DIR / "output" / "LPR_Redux.table_report.all_ts.csv",
    LPR_PYCAP_BASE_DIR / "output" / "LPR_Redux.table_report.base_stream_depletion.csv",
    LPR_PYCAP_INPUTS_DIR / "LPR_Prepped.xlsx",
    LPR_PYCAP_INPUTS_DIR / "LPR_Prepped_IDW.xlsx",
    LPR_PYCAP_INPUTS_DIR / "LPR_Init.xlsx",
    LPR_PYCAP_INPUTS_DIR / "LPR2025.xlsx",
    LPR_PYCAP_INPUTS_DIR / "fish_curves" / "Brook.csv",
    LPR_PYCAP_INPUTS_DIR / "fish_curves" / "Brown.csv",
    LPR_PYCAP_INPUTS_DIR / "fish_curves" / "Rainbow.csv",
]

pycap_file_check = pd.DataFrame(
    [
        {
            "file": path.name,
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
            "size_MB": round(path.stat().st_size / 1_000_000, 3) if path.exists() else None,
        }
        for path in required_pycap_files
    ]
)

pycap_file_check


,file,relative_path,exists,size_MB
0,LPR_Redux.yml,data/raw/lpr_pycap/pycap_base/LPR_Redux.yml,True,0.108
1,depletion_potential.json,data/raw/lpr_pycap/pycap_base/depletion_potent...,True,0.093
2,LPR_Redux.report.txt,data/raw/lpr_pycap/pycap_base/output/LPR_Redux...,True,0.149
3,LPR_Redux.table_report.csv,data/raw/lpr_pycap/pycap_base/output/LPR_Redux...,True,0.141
4,LPR_Redux.table_report.all_ts.csv,data/raw/lpr_pycap/pycap_base/output/LPR_Redux...,True,13.149
5,LPR_Redux.table_report.base_stream_depletion.csv,data/raw/lpr_pycap/pycap_base/output/LPR_Redux...,True,0.009
6,LPR_Prepped.xlsx,data/raw/lpr_pycap/inputs/LPR_Prepped.xlsx,True,0.053
7,LPR_Prepped_IDW.xlsx,data/raw/lpr_pycap/inputs/LPR_Prepped_IDW.xlsx,True,0.051
8,LPR_Init.xlsx,data/raw/lpr_pycap/inputs/LPR_Init.xlsx,True,0.101
9,LPR2025.xlsx,data/raw/lpr_pycap/inputs/LPR2025.xlsx,True,3.360


## 6. Check Python package imports

This checks whether the main scientific Python packages can be imported in the current Jupyter kernel.

A failed package here usually means the wrong kernel/environment is selected.


In [6]:
def check_import(module_name):
    """Try importing a module and return a status dictionary."""
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "unknown")
        return {
            "module": module_name,
            "available": True,
            "version": version,
            "error": "",
        }
    except Exception as err:
        return {
            "module": module_name,
            "available": False,
            "version": None,
            "error": str(err),
        }

modules_to_check = [
    "numpy",
    "pandas",
    "matplotlib",
    "flopy",
    "pyemu",
    "geopandas",
    "shapely",
    "rasterio",
    "openpyxl",
    "yaml",
]

package_check = pd.DataFrame([check_import(m) for m in modules_to_check])
package_check


,module,available,version,error
0,numpy,True,2.4.3,
1,pandas,True,2.3.3,
2,matplotlib,True,3.10.8,
3,flopy,True,3.10.0,
4,pyemu,True,1.4.0,
5,geopandas,True,1.1.3,
6,shapely,True,2.1.2,
7,rasterio,True,1.4.4,
8,openpyxl,True,3.1.5,
9,yaml,True,6.0.3,


## 7. Check possible PyCAP import names

`pycap-dss` may not import with the same name as the package shown in `environment.yml`. This cell checks a few common possibilities and reports which one works.


In [7]:
pycap_candidates = [
    "pycap",
    "pycap_dss",
    "pycapdss",
]

pycap_import_check = pd.DataFrame([check_import(m) for m in pycap_candidates])
pycap_import_check


,module,available,version,error
0,pycap,True,unknown,
1,pycap_dss,False,None,No module named 'pycap_dss'
2,pycapdss,False,None,No module named 'pycapdss'


## 8. Check command-line executables

This checks whether MODFLOW 6 and common PEST++ executables can be found from the active environment.

For this notebook, it is okay if PEST++ executables are missing. We mainly need `mf6` before running the MODFLOW model.


In [8]:
executables_to_check = [
    "mf6",
    "pestpp-mou",
    "pestpp-ies",
    "pestpp-glm",
]

exe_check = pd.DataFrame(
    [
        {
            "executable": exe,
            "found": shutil.which(exe) is not None,
            "path": shutil.which(exe),
        }
        for exe in executables_to_check
    ]
)

exe_check


,executable,found,path
0,mf6,True,/opt/conda/envs/gw_uncertainty/bin/mf6
1,pestpp-mou,True,/opt/conda/envs/gw_uncertainty/bin/pestpp-mou
2,pestpp-ies,True,/opt/conda/envs/gw_uncertainty/bin/pestpp-ies
3,pestpp-glm,True,/opt/conda/envs/gw_uncertainty/bin/pestpp-glm


## 9. Read PyCAP baseline outputs

Now we check whether Python can read the baseline PyCAP output tables.

This does **not** rerun PyCAP. It only reads the files that were copied from the original LPR repository.


In [9]:
pycap_output_csv = LPR_PYCAP_BASE_DIR / "output" / "LPR_Redux.table_report.csv"
pycap_all_ts_csv = LPR_PYCAP_BASE_DIR / "output" / "LPR_Redux.table_report.all_ts.csv"
pycap_base_stream_csv = LPR_PYCAP_BASE_DIR / "output" / "LPR_Redux.table_report.base_stream_depletion.csv"

output_tables = {}

for label, path in {
    "table_report": pycap_output_csv,
    "all_ts": pycap_all_ts_csv,
    "base_stream_depletion": pycap_base_stream_csv,
}.items():
    if path.exists():
        df = pd.read_csv(path)
        output_tables[label] = df
        print(f"{label}: shape = {df.shape}")
        print(f"columns = {list(df.columns)[:10]}")
        print("-" * 80)
    else:
        print(f"Missing: {path}")

# Preview the main table if available
output_tables.get("table_report", pd.DataFrame()).head()


table_report: shape = (330, 329)
columns = ['Unnamed: 0', 'LakeEmily:dd (ft)', 'LPR:466:depl (cfs)', 'LPR:467:depl (cfs)', 'LPR:490:depl (cfs)', 'LPR:509:depl (cfs)', 'LPR:602:depl (cfs)', 'LPR:603:depl (cfs)', 'LPR:798:depl (cfs)', 'LPR:807:depl (cfs)']
--------------------------------------------------------------------------------
all_ts: shape = (1825, 328)
columns = ['Unnamed: 0', 'LPR:466', 'LPR:467', 'LPR:490', 'LPR:509', 'LPR:602', 'LPR:603', 'LPR:798', 'LPR:807', 'LPR:850']
--------------------------------------------------------------------------------
base_stream_depletion: shape = (330, 2)
columns = ['Unnamed: 0', 'LPR']
--------------------------------------------------------------------------------


,Unnamed: 0,LakeEmily:dd (ft),LPR:466:depl (cfs),LPR:467:depl (cfs),LPR:490:depl (cfs),LPR:509:depl (cfs),LPR:602:depl (cfs),LPR:603:depl (cfs),LPR:798:depl (cfs),LPR:807:depl (cfs),...,LPR:93349:depl (cfs),LPR:93422:depl (cfs),LPR:93423:depl (cfs),LPR:93424:depl (cfs),LPR:93469:depl (cfs),LPR:93832:depl (cfs),LPR:94302:depl (cfs),LPR:94988:depl (cfs),LPR:95068:depl (cfs),LPR:418:depl (cfs)
0,418: proposed,6.294692e-138,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00669
1,466: existing,NaN,0.383091,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,467: existing,NaN,NaN,0.28748,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,490: existing,NaN,NaN,NaN,5.003177e-12,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,509: existing,NaN,NaN,NaN,NaN,2.753485e-07,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 10. Read the PyCAP YAML and depletion-potential JSON

This gives a light inventory of the baseline PyCAP configuration and depletion-potential file.


In [10]:
import yaml

pycap_yaml_path = LPR_PYCAP_BASE_DIR / "LPR_Redux.yml"
depletion_json_path = LPR_PYCAP_BASE_DIR / "depletion_potential.json"

if pycap_yaml_path.exists():
    with open(pycap_yaml_path, "r") as f:
        pycap_config = yaml.safe_load(f)
    print("YAML loaded successfully.")
    print(f"Top-level YAML type: {type(pycap_config)}")
    if isinstance(pycap_config, dict):
        print("Top-level YAML keys:")
        for key in list(pycap_config.keys())[:25]:
            print(f"  - {key}")
else:
    print(f"Missing YAML file: {pycap_yaml_path}")

print("\n" + "-" * 80 + "\n")

if depletion_json_path.exists():
    with open(depletion_json_path, "r") as f:
        depletion_data = json.load(f)
    print("depletion_potential.json loaded successfully.")
    print(f"Top-level JSON type: {type(depletion_data)}")
    if isinstance(depletion_data, dict):
        print("Top-level JSON keys:")
        for key in list(depletion_data.keys())[:25]:
            print(f"  - {key}")
else:
    print(f"Missing JSON file: {depletion_json_path}")


YAML loaded successfully.
Top-level YAML type: <class 'dict'>
Top-level YAML keys:
  - project_properties
  - well_418
  - well_466
  - well_467
  - well_490
  - well_509
  - well_602
  - well_603
  - well_798
  - well_807
  - well_850
  - well_862
  - well_1013
  - well_1302
  - well_1323
  - well_1486
  - well_1584
  - well_1589
  - well_1643
  - well_1683
  - well_1860
  - well_2544
  - well_2750
  - well_2886
  - well_3473

--------------------------------------------------------------------------------

depletion_potential.json loaded successfully.
Top-level JSON type: <class 'dict'>
Top-level JSON keys:
  - type
  - name
  - crs
  - features


## 11. Try loading the MODFLOW 6 simulation with FloPy

This only loads the model structure. It does **not** run the model.

If this fails, it usually means one of the MODFLOW input paths is missing or the working directory is wrong.


In [11]:
try:
    import flopy

    sim = flopy.mf6.MFSimulation.load(
        sim_ws=str(LPR_MF6_DIR),
        verbosity_level=1,
    )

    print("MODFLOW 6 simulation loaded successfully with FloPy.")
    print(f"Simulation name: {sim.name}")
    print(f"Model names: {sim.model_names}")

    gwf = sim.get_model()
    print(f"Groundwater model name: {gwf.name}")
    print("Packages:")
    for package in gwf.packagelist:
        print(f"  - {package.package_type}: {package.filename}")

except Exception as err:
    print("FloPy could not load the MODFLOW 6 simulation.")
    print(type(err).__name__)
    print(err)


loading simulation...
  loading simulation name file...
  loading tdis package...
  loading model gwf6...
    loading package dis...
    loading package sfr...
    loading package ic...
    loading package npf...
    loading package oc...
    loading package rch...
    loading package chd...
    loading package drn...
    loading package wel...
  loading solution package lpr_ss_gwf...
MODFLOW 6 simulation loaded successfully with FloPy.
Simulation name: modflowsim
Model names: ['lpr_ss_gwf']
Groundwater model name: lpr_ss_gwf
Packages:
  - dis: lpr_ss_gwf.dis
  - obs: sfr_obs.obs
  - sfr: lpr_ss.sfr
  - ic: lpr_ss_gwf.ic
  - npf: lpr_ss_gwf.npf
  - oc: lpr_ss_gwf.oc
  - rcha: lpr_ss_gwf.rcha
  - chd: lpr_ss_gwf.chd
  - drn: lpr_ss_gwf.drn
  - wel: lpr_ss_gwf.wel


## 12. Optional: check the MODFLOW 6 executable version

This is useful before running the model.


In [12]:
mf6_path = shutil.which("mf6")

if mf6_path is None:
    print("mf6 executable was not found in the active environment.")
else:
    print(f"mf6 found at: {mf6_path}")
    try:
        result = subprocess.run(
            ["mf6", "-v"],
            capture_output=True,
            text=True,
            timeout=20,
        )
        print("STDOUT:")
        print(result.stdout)
        print("STDERR:")
        print(result.stderr)
    except Exception as err:
        print("Could not get mf6 version.")
        print(type(err).__name__)
        print(err)


mf6 found at: /opt/conda/envs/gw_uncertainty/bin/mf6
STDOUT:
 
mf6: 6.6.3 09/29/2025
 

STDERR:



## 13. Create a summary check table

This creates a combined status table and saves it to:

```text
results/environment_and_file_check.csv
```


In [13]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

summary_rows = []

def add_summary(category, name, passed, note=""):
    summary_rows.append(
        {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "category": category,
            "item": name,
            "passed": bool(passed),
            "note": note,
        }
    )

for _, row in dir_check.iterrows():
    add_summary("directory", row["item"], row["exists"], row["path"])

for _, row in mf6_file_check.iterrows():
    add_summary("mf6_file", row["file"], row["exists"], row["relative_path"])

for _, row in pycap_file_check.iterrows():
    add_summary("pycap_file", row["file"], row["exists"], row["relative_path"])

for _, row in package_check.iterrows():
    add_summary("python_package", row["module"], row["available"], row["version"] or row["error"])

for _, row in pycap_import_check.iterrows():
    add_summary("pycap_import_candidate", row["module"], row["available"], row["version"] or row["error"])

for _, row in exe_check.iterrows():
    add_summary("executable", row["executable"], row["found"], row["path"] or "not found")

summary_check = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / "environment_and_file_check.csv"
summary_check.to_csv(summary_path, index=False)

print(f"Saved summary check to: {summary_path.relative_to(PROJECT_ROOT)}")
summary_check


Saved summary check to: results/environment_and_file_check.csv


,timestamp,category,item,passed,note
0,2026-04-19T04:00:41,directory,Project root,True,.
1,2026-04-19T04:00:41,directory,MODFLOW 6 steady-state model,True,models/lpr_mf6/steady_state
2,2026-04-19T04:00:41,directory,PyCAP baseline folder,True,data/raw/lpr_pycap/pycap_base
3,2026-04-19T04:00:41,directory,PyCAP inputs folder,True,data/raw/lpr_pycap/inputs
4,2026-04-19T04:00:41,directory,Legacy scripts folder,True,scripts/lpr_legacy
5,2026-04-19T04:00:41,directory,Results folder,True,results
6,2026-04-19T04:00:41,directory,Figures folder,True,figures
7,2026-04-19T04:00:41,mf6_file,mfsim.nam,True,models/lpr_mf6/steady_state/mfsim.nam
8,2026-04-19T04:00:41,mf6_file,lpr_ss.tdis,True,models/lpr_mf6/steady_state/lpr_ss.tdis
9,2026-04-19T04:00:41,mf6_file,lpr_ss.ims,True,models/lpr_mf6/steady_state/lpr_ss.ims


## 14. Final interpretation

Use this checklist:

- If all required directories are `True`, the project layout is good.
- If all required MODFLOW files are `True`, the MF6 model files copied correctly.
- If all required PyCAP files are `True`, the PyCAP baseline/input files copied correctly.
- If `flopy`, `pandas`, `numpy`, `yaml`, and `openpyxl` import correctly, the notebook can inspect the model files.
- If `mf6` is found, the next notebook can attempt a MODFLOW run.
- If PyCAP does not import yet, that is okay for this notebook, but we will fix it before trying to rerun PyCAP.

Recommended next notebook:

```text
01_mf6_baseline_run_check.ipynb
```
